# ResNet R1-R7 Experiment Matrix

Runs the planned ResNet comparison experiments with Macro-F1 checkpointing and exports one CSV for the report.


## 1. Setup


In [ ]:
# ============================================================
# Imports
# Keep all third-party imports in one place so missing packages fail early.
# ============================================================

import os
import gc
import json
import time
import random
import numpy as np
from pathlib import Path
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageOps

try:
    import cv2
except ImportError:
    cv2 = None
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, classification_report

import torch.nn as nn
import torch.optim as optim
from torchvision import models
from tqdm import tqdm


In [ ]:
# ============================================================
# Configuration
# This notebook is prepared for the planned ResNet R1-R7 comparison runs.
# The experiment matrix below controls the individual runs.
# ============================================================

DATA_ROOT = Path("/kaggle/input/datasets/paulconrad21/eye-disease-dataset/eye-disease-data")

TRAIN_CSV_PATH = DATA_ROOT / "2. Groundtruths" / "a. IDRiD_Disease Grading_Training Labels.csv"
TEST_CSV_PATH  = DATA_ROOT / "2. Groundtruths" / "b. IDRiD_Disease Grading_Testing Labels.csv"
TRAIN_IMAGE_DIR = DATA_ROOT / "1. Original Images" / "a. Training Set"
TEST_IMAGE_DIR  = DATA_ROOT / "1. Original Images" / "b. Testing Set"

IMAGE_COL = "Image name"
LABEL_COL = "Retinopathy grade"
NUM_CLASSES = 5
DEVICE_MODE = "cuda"

IMAGE_SIZE = 384
NUM_WORKERS = 2
NUM_EPOCHS = 60
PATIENCE = 12
MIN_DELTA = 1e-4
SELECTION_METRIC = "macro_f1"
SEED = 42

RESULTS_CSV_PATH = "/kaggle/working/resnet_r1_r7_experiment_results.csv"
EPOCH_HISTORY_CSV_PATH = "/kaggle/working/resnet_r1_r7_epoch_history.csv"
BEST_MODEL_PATH = "/kaggle/working/best_resnet_model.pth"
EXPERIMENT_NAME = "resnet_r1_r7_experiment_matrix"

RUN_EXPERIMENT_MATRIX = True
RUN_BASELINE_TRAINING = False
RUN_FINAL_EVALUATION = False
RUN_GRADCAM = False
SHOW_DATA_PREVIEW = False

RUN_OPTUNA = False
OPTUNA_N_TRIALS = 8
OPTUNA_MAX_EPOCHS = 12
OPTUNA_PATIENCE = 4

USE_WANDB = False
WANDB_PROJECT = "Practical_Deep_Learning_Course"
WANDB_GROUP = "resnet_idrid_experiments"

BATCH_SIZE = 4
USE_FUNDUS_PREPROCESSING = True
PREPROCESSING_PRESET = "none"
PREPROCESS_CROP_BLACK_BACKGROUND = True
PREPROCESS_AUTOCONTRAST = False
PREPROCESS_CROP_THRESHOLD = 10
PREPROCESS_CROP_PADDING = 16
PREPROCESS_CIRCULAR_MASK = False
PREPROCESS_GREEN_CHANNEL = False
PREPROCESS_CLAHE = False
PREPROCESS_CLAHE_MODE = "lab"
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID_SIZE = (8, 8)
PREPROCESS_GRAHAM = False
GRAHAM_BLUR_SIGMA = 10
GRAHAM_BLEND_ALPHA = 4.0
GRAHAM_BLEND_BETA = -4.0
GRAHAM_BLEND_GAMMA = 128

USE_FROZEN_BACKBONE = True
USE_CUSTOM_CLASSIFICATION_HEAD = True
CUSTOM_HEAD_HIDDEN_DIM = 512
RESNET_DROPOUT = 0.20

USE_AUGMENTATION = False
USE_RANDOM_FLIP = False
USE_RANDOM_ROTATION = False
USE_COLOR_JITTER = False
USE_RANDOM_RESIZED_CROP = False
USE_RANDOM_ERASING = False
RANDOM_ROTATION_DEGREES = 10
RANDOM_RESIZED_CROP_SCALE = (0.85, 1.0)
COLOR_JITTER_BRIGHTNESS = 0.10
COLOR_JITTER_CONTRAST = 0.10
COLOR_JITTER_SATURATION = 0.05
COLOR_JITTER_HUE = 0.02
RANDOM_ERASING_P = 0.15

LOSS_NAME = "cross_entropy"
FOCAL_GAMMA = 2.0
OPTIMIZER_NAME = "adamw"
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-3
MOMENTUM = 0.90
USE_LR_SCHEDULER = False
LR_SCHEDULER_FACTOR = 0.5
LR_SCHEDULER_PATIENCE = 3
MIN_LR = 1e-6
SAMPLING_STRATEGY = "none"
VAL_SIZE = 0.2
USE_STRATIFIED_SPLIT = True

USE_WEIGHTED_LOSS = LOSS_NAME in ["weighted_cross_entropy", "weighted_focal_loss"]
USE_WEIGHTED_SAMPLER = SAMPLING_STRATEGY == "weighted_oversample"

BASE_RUN_CONFIG = {
    "model_name": "resnet18",
    "image_size": IMAGE_SIZE,
    "batch_size": 4,
    "num_epochs": NUM_EPOCHS,
    "patience": PATIENCE,
    "selection_metric": "macro_f1",
    "optimizer_name": "adamw",
    "learning_rate": 1e-4,
    "weight_decay": 1e-3,
    "momentum": 0.90,
    "use_lr_scheduler": False,
    "use_frozen_backbone": True,
    "use_custom_classification_head": True,
    "custom_head_hidden_dim": 512,
    "resnet_dropout": 0.20,
    "sampling_strategy": "none",
    "use_augmentation": False,
    "use_random_flip": False,
    "use_random_rotation": False,
    "use_color_jitter": False,
    "use_random_resized_crop": False,
    "use_random_erasing": False,
    "loss_name": "cross_entropy",
    "focal_gamma": 2.0,
}

EXPERIMENT_RUNS = [
    {**BASE_RUN_CONFIG, "run_id": "R1", "experiment_name": "R1_resnet_none_ce_frozen", "preprocessing_preset": "none", "report_note": "Baseline: no special preprocessing, no sampling, CE loss."},
    {**BASE_RUN_CONFIG, "run_id": "R2", "experiment_name": "R2_resnet_none_weighted_ce_frozen", "preprocessing_preset": "none", "loss_name": "weighted_cross_entropy", "report_note": "Tests class-weighted loss without changing data sampling."},
    {**BASE_RUN_CONFIG, "run_id": "R3", "experiment_name": "R3_resnet_crop_graham_ce_frozen", "preprocessing_preset": "crop_graham", "report_note": "Tests Ben-Graham-style illumination correction."},
    {**BASE_RUN_CONFIG, "run_id": "R4", "experiment_name": "R4_resnet_crop_circular_autocontrast_ce_frozen", "preprocessing_preset": "crop_circular_autocontrast", "report_note": "Tests circular masking plus autocontrast after ROI crop."},
    {**BASE_RUN_CONFIG, "run_id": "R5", "experiment_name": "R5_resnet_crop_graham_oversample_ce_frozen", "preprocessing_preset": "crop_graham", "sampling_strategy": "weighted_oversample", "report_note": "Tests probabilistic oversampling of rare classes."},
    {**BASE_RUN_CONFIG, "run_id": "R6", "experiment_name": "R6_resnet_crop_graham_mild_aug_ce_frozen", "preprocessing_preset": "crop_graham", "use_augmentation": True, "use_random_flip": True, "use_random_rotation": True, "use_color_jitter": True, "report_note": "Tests mild medical-image-safe augmentation."},
    {**BASE_RUN_CONFIG, "run_id": "R7", "experiment_name": "R7_resnet_crop_graham_ce_finetune", "preprocessing_preset": "crop_graham", "use_frozen_backbone": False, "learning_rate": 3e-5, "weight_decay": 1e-4, "report_note": "Tests full fine-tuning instead of frozen backbone with a smaller LR."},
]


def get_device():
    if DEVICE_MODE == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if DEVICE_MODE == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but not available. Check Kaggle accelerator settings.")
        return torch.device("cuda")
    if DEVICE_MODE == "cpu":
        return torch.device("cpu")
    raise ValueError(f"Invalid DEVICE_MODE: {DEVICE_MODE}")

DEVICE = get_device()


In [ ]:
# ============================================================
# Reproducibility
# Fixed seeds make train/validation splits and training behavior easier to compare.
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# Config validation
# Fail fast if a config combination is invalid or paths are missing.
# ============================================================

def validate_config():

    valid_losses = [
        "cross_entropy",
        "weighted_cross_entropy",
        "focal_loss",
        "weighted_focal_loss",
    ]

    if LOSS_NAME not in valid_losses:
        raise ValueError(f"Invalid LOSS_NAME: {LOSS_NAME}")

    valid_sampling_strategies = ["none", "weighted_oversample", "undersample"]
    if SAMPLING_STRATEGY not in valid_sampling_strategies:
        raise ValueError(f"Invalid SAMPLING_STRATEGY: {SAMPLING_STRATEGY}")

    valid_optimizers = ["adam", "adamw", "sgd"]

    if OPTIMIZER_NAME not in valid_optimizers:
        raise ValueError(f"Invalid OPTIMIZER_NAME: {OPTIMIZER_NAME}")

    if IMAGE_SIZE <= 0:
        raise ValueError("IMAGE_SIZE must be > 0")

    if BATCH_SIZE <= 0:
        raise ValueError("BATCH_SIZE must be > 0")

    if NUM_EPOCHS <= 0:
        raise ValueError("NUM_EPOCHS must be > 0")

    if PATIENCE <= 0:
        raise ValueError("PATIENCE must be > 0")

    if MIN_DELTA < 0:
        raise ValueError("MIN_DELTA must be >= 0")

    valid_selection_metrics = ["accuracy", "kappa", "macro_f1"]
    if SELECTION_METRIC not in valid_selection_metrics:
        raise ValueError(f"Invalid SELECTION_METRIC: {SELECTION_METRIC}")

    valid_preprocessing_presets = [
        "none",
        "crop_only",
        "crop_autocontrast",
        "crop_clahe_lab",
        "crop_clahe_green",
        "crop_green",
        "crop_graham",
        "crop_circular_autocontrast",
    ]
    if PREPROCESSING_PRESET not in valid_preprocessing_presets:
        raise ValueError(f"Invalid PREPROCESSING_PRESET: {PREPROCESSING_PRESET}")

    for run_config in EXPERIMENT_RUNS:
        if run_config["preprocessing_preset"] not in valid_preprocessing_presets:
            raise ValueError(f"Invalid preprocessing preset in {run_config['run_id']}: {run_config['preprocessing_preset']}")

    if PREPROCESS_CROP_THRESHOLD < 0:
        raise ValueError("PREPROCESS_CROP_THRESHOLD must be >= 0")

    if PREPROCESS_CROP_PADDING < 0:
        raise ValueError("PREPROCESS_CROP_PADDING must be >= 0")

    if CUSTOM_HEAD_HIDDEN_DIM <= 0:
        raise ValueError("CUSTOM_HEAD_HIDDEN_DIM must be > 0")

    if not 0 <= RESNET_DROPOUT < 1:
        raise ValueError("RESNET_DROPOUT must be >= 0 and < 1")

    if RANDOM_ROTATION_DEGREES < 0:
        raise ValueError("RANDOM_ROTATION_DEGREES must be >= 0")

    if not 0 <= RANDOM_ERASING_P <= 1:
        raise ValueError("RANDOM_ERASING_P must be between 0 and 1")

    if not 0 < LR_SCHEDULER_FACTOR < 1:
        raise ValueError("LR_SCHEDULER_FACTOR must be > 0 and < 1")

    if LR_SCHEDULER_PATIENCE <= 0:
        raise ValueError("LR_SCHEDULER_PATIENCE must be > 0")

    if MIN_LR <= 0:
        raise ValueError("MIN_LR must be > 0")

    if NUM_CLASSES <= 1:
        raise ValueError("NUM_CLASSES must be > 1")

    if not 0 < VAL_SIZE < 1:
        raise ValueError("VAL_SIZE must be between 0 and 1")

    if RUN_OPTUNA:
        if OPTUNA_N_TRIALS <= 0:
            raise ValueError("OPTUNA_N_TRIALS must be > 0")
        if OPTUNA_MAX_EPOCHS <= 0:
            raise ValueError("OPTUNA_MAX_EPOCHS must be > 0")
        if OPTUNA_PATIENCE <= 0:
            raise ValueError("OPTUNA_PATIENCE must be > 0")

    # Check paths exist.
    if not DATA_ROOT.exists():
        raise FileNotFoundError(f"DATA_ROOT not found: {DATA_ROOT}")

    if not TRAIN_IMAGE_DIR.exists():
        raise FileNotFoundError(f"TRAIN_IMAGE_DIR not found: {TRAIN_IMAGE_DIR}")

    if not TRAIN_CSV_PATH.exists():
        raise FileNotFoundError(f"TRAIN_CSV_PATH not found: {TRAIN_CSV_PATH}")

    if not TEST_CSV_PATH.exists():
        raise FileNotFoundError(f"TEST_CSV_PATH not found: {TEST_CSV_PATH}")

    if not TEST_IMAGE_DIR.exists():
        raise FileNotFoundError(f"TEST_IMAGE_DIR not found: {TEST_IMAGE_DIR}")


# ============================================================
# Print config
# Printing the full setup helps document which experiment produced which result.
# ============================================================

def print_config():

    print("=" * 60)
    print("EXPERIMENT CONFIG")
    print("=" * 60)

    print("\nDATA PATHS:")
    print(f"  DATA_ROOT: {DATA_ROOT}")
    print(f"  TRAIN_CSV_PATH: {TRAIN_CSV_PATH}")
    print(f"  TEST_CSV_PATH: {TEST_CSV_PATH}")
    print(f"  TEST_IMAGE_DIR: {TEST_IMAGE_DIR}")

    print("\nDATASET:")
    print(f"  IMAGE_COL: {IMAGE_COL}")
    print(f"  LABEL_COL: {LABEL_COL}")
    print(f"  NUM_CLASSES: {NUM_CLASSES}")

    print("\nDEVICE:")
    print(f"  DEVICE_MODE: {DEVICE_MODE}")
    print(f"  DEVICE: {DEVICE}")

    print("\nIMAGE:")
    print(f"  IMAGE_SIZE: {IMAGE_SIZE}")
    print(f"  PREPROCESSING_PRESET: {PREPROCESSING_PRESET}")

    print("\nMATRIX:")
    print(f"  RUN_EXPERIMENT_MATRIX: {RUN_EXPERIMENT_MATRIX}")
    print(f"  Planned runs: {[run['run_id'] for run in EXPERIMENT_RUNS]}")

    print("\nTRAINING:")
    print(f"  BATCH_SIZE: {BATCH_SIZE}")
    print(f"  NUM_WORKERS: {NUM_WORKERS}")
    print(f"  NUM_EPOCHS: {NUM_EPOCHS}")
    print(f"  PATIENCE: {PATIENCE}")
    print(f"  MIN_DELTA: {MIN_DELTA}")
    print(f"  SELECTION_METRIC: {SELECTION_METRIC}")
    print(f"  SEED: {SEED}")
    print(f"  BEST_MODEL_PATH: {BEST_MODEL_PATH}")
    print(f"  RESULTS_CSV_PATH: {RESULTS_CSV_PATH}")
    print(f"  EXPERIMENT_NAME: {EXPERIMENT_NAME}")

    print("\nEXPERIMENT TRACKING:")
    print(f"  USE_WANDB: {USE_WANDB}")
    print(f"  WANDB_PROJECT: {WANDB_PROJECT}")
    print(f"  WANDB_GROUP: {WANDB_GROUP}")

    print("\nOPTUNA:")
    print(f"  RUN_OPTUNA: {RUN_OPTUNA}")
    print(f"  RUN_BASELINE_TRAINING: {RUN_BASELINE_TRAINING}")
    print(f"  RUN_FINAL_EVALUATION: {RUN_FINAL_EVALUATION}")
    print(f"  RUN_GRADCAM: {RUN_GRADCAM}")
    print(f"  OPTUNA_N_TRIALS: {OPTUNA_N_TRIALS}")
    print(f"  OPTUNA_MAX_EPOCHS: {OPTUNA_MAX_EPOCHS}")
    print(f"  OPTUNA_PATIENCE: {OPTUNA_PATIENCE}")

    print("\nMODEL REGULARIZATION:")
    print(f"  RESNET_DROPOUT: {RESNET_DROPOUT}")

    print("\nAUGMENTATION:")
    print(f"  USE_AUGMENTATION: {USE_AUGMENTATION}")
    print(f"  FLIP: {USE_RANDOM_FLIP}")
    print(f"  ROTATION: {USE_RANDOM_ROTATION}")
    print(f"  COLOR_JITTER: {USE_COLOR_JITTER}")
    print(f"  RANDOM_CROP: {USE_RANDOM_RESIZED_CROP}")

    print("\nIMBALANCE HANDLING:")
    print(f"  USE_WEIGHTED_LOSS: {USE_WEIGHTED_LOSS}")
    print(f"  USE_WEIGHTED_SAMPLER: {USE_WEIGHTED_SAMPLER}")

    print("\nLOSS:")
    print(f"  LOSS_NAME: {LOSS_NAME}")
    print(f"  FOCAL_GAMMA: {FOCAL_GAMMA}")

    print("\nOPTIMIZER:")
    print(f"  OPTIMIZER_NAME: {OPTIMIZER_NAME}")
    print(f"  LEARNING_RATE: {LEARNING_RATE}")
    print(f"  WEIGHT_DECAY: {WEIGHT_DECAY}")
    print(f"  MOMENTUM: {MOMENTUM}")

    print("\nLR SCHEDULER:")
    print(f"  USE_LR_SCHEDULER: {USE_LR_SCHEDULER}")
    print(f"  LR_SCHEDULER_FACTOR: {LR_SCHEDULER_FACTOR}")
    print(f"  LR_SCHEDULER_PATIENCE: {LR_SCHEDULER_PATIENCE}")
    print(f"  MIN_LR: {MIN_LR}")

    print("\nVALIDATION:")
    print(f"  VAL_SIZE: {VAL_SIZE}")
    print(f"  STRATIFIED: {USE_STRATIFIED_SPLIT}")

    print("=" * 60)


# ============================================================
# RUN CHECKS
# ============================================================

validate_config()
print_config()


## 2. Data Preparation


In [ ]:
# ============================================================
# Load CSV files
# The official IDRiD train and test CSVs are kept separate; the test set is only used at the end.
# ============================================================

train_full_df = pd.read_csv(TRAIN_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)

print("Training CSV columns:")
print(train_full_df.columns.tolist())

print("\nTesting CSV columns:")
print(test_df.columns.tolist())

print("\nTraining CSV preview:")
display(train_full_df.head())

print("\nTesting CSV preview:")
display(test_df.head())


# ============================================================
# Select task columns
# Keep only image IDs and DR severity labels for this classification task.
# ============================================================

IMAGE_COL = "Image name"
LABEL_COL = "Retinopathy grade"

# Keep only the columns needed for this classification task
train_full_df = train_full_df[[IMAGE_COL, LABEL_COL]].copy()
test_df = test_df[[IMAGE_COL, LABEL_COL]].copy()

# Make sure labels are integers
train_full_df[LABEL_COL] = train_full_df[LABEL_COL].astype(int)
test_df[LABEL_COL] = test_df[LABEL_COL].astype(int)

print("Using image column:", IMAGE_COL)
print("Using label column:", LABEL_COL)

print("\nTrain class distribution:")
print(train_full_df[LABEL_COL].value_counts().sort_index())

print("\nTest class distribution:")
print(test_df[LABEL_COL].value_counts().sort_index())


In [ ]:
# ============================================================
# Dataset class
# Converts CSV rows into image tensors and integer labels.
# ============================================================

def crop_black_background(image, threshold=PREPROCESS_CROP_THRESHOLD, padding=PREPROCESS_CROP_PADDING):
    """
    Crops large black borders around fundus photographs.

    This approximates retinal ROI cropping and removes uninformative background.
    """

    gray = np.array(image.convert("L"))
    mask = gray > threshold

    if not mask.any():
        return image

    ys, xs = np.where(mask)

    left = max(int(xs.min()) - padding, 0)
    right = min(int(xs.max()) + padding + 1, image.width)
    top = max(int(ys.min()) - padding, 0)
    bottom = min(int(ys.max()) + padding + 1, image.height)

    if right <= left or bottom <= top:
        return image

    return image.crop((left, top, right, bottom))


def apply_circular_mask(image):
    """
    Masks pixels outside the central retinal circle.

    This can reduce residual corner/background artifacts after ROI cropping.
    """

    arr = np.array(image).copy()
    height, width = arr.shape[:2]
    center_x, center_y = width // 2, height // 2
    radius = int(0.5 * min(width, height))

    yy, xx = np.ogrid[:height, :width]
    mask = (xx - center_x) ** 2 + (yy - center_y) ** 2 <= radius ** 2
    arr[~mask] = 0

    return Image.fromarray(arr)


def apply_green_channel(image):
    """
    Keeps the green channel and repeats it to RGB.

    The green channel often provides strong contrast for vessels and lesions in fundus images
    while preserving the 3-channel input expected by ImageNet-pretrained CNNs.
    """

    arr = np.array(image)
    green = arr[:, :, 1]
    green_rgb = np.stack([green, green, green], axis=-1)

    return Image.fromarray(green_rgb.astype(np.uint8))


def apply_clahe(image, mode=PREPROCESS_CLAHE_MODE):
    """
    Applies CLAHE contrast enhancement.

    mode="lab"   -> apply CLAHE to LAB lightness channel, preserving color best
    mode="rgb"   -> apply CLAHE independently to RGB channels
    mode="green" -> apply CLAHE to green channel and repeat to RGB
    """

    if cv2 is None:
        raise ImportError("OpenCV/cv2 is required for CLAHE preprocessing.")

    arr = np.array(image)
    clahe = cv2.createCLAHE(
        clipLimit=CLAHE_CLIP_LIMIT,
        tileGridSize=CLAHE_TILE_GRID_SIZE
    )

    if mode == "lab":
        lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    elif mode == "rgb":
        channels = cv2.split(arr)
        enhanced_channels = [clahe.apply(channel) for channel in channels]
        enhanced = cv2.merge(enhanced_channels)

    elif mode == "green":
        green = clahe.apply(arr[:, :, 1])
        enhanced = np.stack([green, green, green], axis=-1)

    else:
        raise ValueError(f"Invalid CLAHE mode: {mode}")

    return Image.fromarray(enhanced.astype(np.uint8))


def apply_graham_preprocessing(image):
    """
    Applies a Ben-Graham-style illumination correction.

    It emphasizes local structures by subtracting a Gaussian-blurred version of the image:
    addWeighted(image, alpha, blur(image), beta, gamma).
    """

    if cv2 is None:
        raise ImportError("OpenCV/cv2 is required for Graham preprocessing.")

    arr = np.array(image)
    blurred = cv2.GaussianBlur(arr, (0, 0), GRAHAM_BLUR_SIGMA)
    enhanced = cv2.addWeighted(
        arr,
        GRAHAM_BLEND_ALPHA,
        blurred,
        GRAHAM_BLEND_BETA,
        GRAHAM_BLEND_GAMMA
    )

    return Image.fromarray(np.clip(enhanced, 0, 255).astype(np.uint8))


def preprocess_fundus_image(image, preset=PREPROCESSING_PRESET):
    """
    Optional deterministic preprocessing before augmentation/resize.

    Presets are intentionally clear recipes rather than arbitrary Boolean mixtures.
    """

    options = get_preprocessing_options(preset)

    if not options["use_preprocessing"]:
        return image

    if options["crop_black_background"]:
        image = crop_black_background(image)

    if options["circular_mask"]:
        image = apply_circular_mask(image)

    if options["green_channel"]:
        image = apply_green_channel(image)

    if options["clahe"]:
        image = apply_clahe(image, mode=options["clahe_mode"])

    if options["graham"]:
        image = apply_graham_preprocessing(image)

    if options["autocontrast"]:
        image = ImageOps.autocontrast(image)

    return image


def get_preprocessing_options(preset=PREPROCESSING_PRESET):
    """
    Converts a named preprocessing preset into concrete preprocessing switches.

    This is used by Optuna so each trial can test a clear, interpretable preprocessing recipe.
    """

    options = {
        "use_preprocessing": True,
        "crop_black_background": True,
        "autocontrast": False,
        "circular_mask": False,
        "green_channel": False,
        "clahe": False,
        "clahe_mode": "lab",
        "graham": False,
    }

    if preset == "none":
        options.update({
            "use_preprocessing": False,
            "crop_black_background": False,
        })
    elif preset == "crop_only":
        pass
    elif preset == "crop_autocontrast":
        options["autocontrast"] = True
    elif preset == "crop_clahe_lab":
        options["clahe"] = True
        options["clahe_mode"] = "lab"
    elif preset == "crop_clahe_green":
        options["clahe"] = True
        options["clahe_mode"] = "green"
    elif preset == "crop_graham":
        options["graham"] = True
    elif preset == "crop_green":
        options["green_channel"] = True
    elif preset == "crop_circular_autocontrast":
        options["circular_mask"] = True
        options["autocontrast"] = True
    else:
        raise ValueError(f"Unknown preprocessing preset: {preset}")

    return options


class FundusDataset(Dataset):
    """
    Dataset for loading fundus images and labels.

    Input:
        dataframe: table with image names and labels
        image_dir: folder containing the images
        transform: preprocessing / augmentation
        image_col: column containing image names
        label_col: column containing labels

    Output per item:
        image tensor, label
    """

    def __init__(self, dataframe, image_dir, transform=None, image_col=None, label_col=None, preprocessing_preset=PREPROCESSING_PRESET):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.image_col = image_col
        self.label_col = label_col
        self.preprocessing_preset = preprocessing_preset

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_name = str(row[self.image_col])

        # Add .jpg if the CSV only contains IDRiD_001 instead of IDRiD_001.jpg.
        if not image_name.lower().endswith((".jpg", ".jpeg", ".png")):
            image_name += ".jpg"

        image_path = self.image_dir / image_name

        image = Image.open(image_path).convert("RGB")
        image = preprocess_fundus_image(image, preset=self.preprocessing_preset)
        label = int(row[self.label_col])

        if self.transform is not None:
            image = self.transform(image)

        return image, label

# ============================================================
# Image transforms
# Optional fundus preprocessing happens in the Dataset before these transforms.
# Train transforms can include augmentation; validation/test transforms stay deterministic.
# ============================================================

def get_transforms(image_size, train=True, use_augmentation=USE_AUGMENTATION):
    """
    Creates image preprocessing.

    Training:
        uses random augmentation if enabled

    Validation/Test:
        deterministic preprocessing only
    """

    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]

    steps = []

    if train and use_augmentation:

        if USE_RANDOM_RESIZED_CROP:
            steps.append(
                transforms.RandomResizedCrop(
                    image_size,
                    scale=RANDOM_RESIZED_CROP_SCALE
                )
            )
        else:
            steps.append(transforms.Resize((image_size, image_size)))

        if USE_RANDOM_FLIP:
            steps.append(transforms.RandomHorizontalFlip())
            steps.append(transforms.RandomVerticalFlip())

        if USE_RANDOM_ROTATION:
            steps.append(transforms.RandomRotation(RANDOM_ROTATION_DEGREES))

        if USE_COLOR_JITTER:
            steps.append(
                transforms.ColorJitter(
                    brightness=COLOR_JITTER_BRIGHTNESS,
                    contrast=COLOR_JITTER_CONTRAST,
                    saturation=COLOR_JITTER_SATURATION,
                    hue=COLOR_JITTER_HUE
                )
            )

    else:
        steps.append(transforms.Resize((image_size, image_size)))

    steps.append(transforms.ToTensor())
    steps.append(transforms.Normalize(mean=imagenet_mean, std=imagenet_std))

    if train and use_augmentation and USE_RANDOM_ERASING:
        steps.append(transforms.RandomErasing(p=RANDOM_ERASING_P))

    return transforms.Compose(steps)


In [ ]:
# ============================================================
# Class imbalance helpers
# Class weights affect the loss; sampling affects which examples are seen during training.
# ============================================================

def compute_class_weights(labels, num_classes):
    """
    Computes class weights for weighted loss.

    Rare classes get higher weights.
    """

    labels = np.array(labels)
    class_counts = np.bincount(labels, minlength=num_classes)

    class_counts = np.maximum(class_counts, 1)
    total_samples = len(labels)
    class_weights = total_samples / (num_classes * class_counts)

    return torch.tensor(class_weights, dtype=torch.float32)


def create_weighted_sampler(labels):
    """
    Creates a WeightedRandomSampler.

    Rare classes are sampled more often during training. This keeps all original samples
    and changes sampling probability instead of physically duplicating rows.
    """

    labels = np.array(labels)
    class_counts = np.bincount(labels, minlength=NUM_CLASSES)
    class_counts = np.maximum(class_counts, 1)

    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels]

    return WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.float32),
        num_samples=len(sample_weights),
        replacement=True
    )


def undersample_dataframe(dataframe, label_col=LABEL_COL, seed=SEED):
    """
    Reduces every class to the minority-class count.

    This can make classes balanced, but it discards data. Use it as an experiment,
    not as a default assumption.
    """

    min_count = dataframe[label_col].value_counts().min()

    balanced_parts = []
    for _, group in dataframe.groupby(label_col):
        balanced_parts.append(
            group.sample(n=min_count, random_state=seed)
        )

    return (
        pd.concat(balanced_parts)
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )


# ============================================================
# Create train / validation / test DataLoaders
# Validation is split from official training data. Official test data remains untouched until final evaluation.
# ============================================================

def create_dataloaders(
    batch_size=BATCH_SIZE,
    sampling_strategy=SAMPLING_STRATEGY,
    use_augmentation=USE_AUGMENTATION,
    preprocessing_preset=PREPROCESSING_PRESET
):
    """
    Creates train/validation/test loaders.

    sampling_strategy:
        "none"                -> original train distribution
        "weighted_oversample" -> probabilistic oversampling via WeightedRandomSampler
        "undersample"         -> physically undersample majority classes in train_df
    """

    if USE_STRATIFIED_SPLIT:
        train_df, val_df = train_test_split(
            train_full_df,
            test_size=VAL_SIZE,
            random_state=SEED,
            stratify=train_full_df[LABEL_COL]
        )
    else:
        train_df, val_df = train_test_split(
            train_full_df,
            test_size=VAL_SIZE,
            random_state=SEED
        )

    if sampling_strategy == "undersample":
        train_df = undersample_dataframe(train_df)

    train_transform = get_transforms(
        IMAGE_SIZE,
        train=True,
        use_augmentation=use_augmentation
    )
    eval_transform = get_transforms(
        IMAGE_SIZE,
        train=False,
        use_augmentation=False
    )

    train_dataset = FundusDataset(
        dataframe=train_df,
        image_dir=TRAIN_IMAGE_DIR,
        transform=train_transform,
        image_col=IMAGE_COL,
        label_col=LABEL_COL,
        preprocessing_preset=preprocessing_preset
    )

    val_dataset = FundusDataset(
        dataframe=val_df,
        image_dir=TRAIN_IMAGE_DIR,
        transform=eval_transform,
        image_col=IMAGE_COL,
        label_col=LABEL_COL,
        preprocessing_preset=preprocessing_preset
    )

    test_dataset = FundusDataset(
        dataframe=test_df,
        image_dir=TEST_IMAGE_DIR,
        transform=eval_transform,
        image_col=IMAGE_COL,
        label_col=LABEL_COL,
        preprocessing_preset=preprocessing_preset
    )

    class_weights = compute_class_weights(
        labels=train_df[LABEL_COL].values,
        num_classes=NUM_CLASSES
    )

    if sampling_strategy == "weighted_oversample":
        train_sampler = create_weighted_sampler(train_df[LABEL_COL].values)
        shuffle_train = False
    else:
        train_sampler = None
        shuffle_train = True

    pin_memory = DEVICE.type == "cuda"

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle_train,
        sampler=train_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory
    )

    return train_loader, val_loader, test_loader, class_weights, train_df, val_df


train_loader, val_loader, test_loader, class_weights, train_df, val_df = create_dataloaders(
    batch_size=BATCH_SIZE,
    sampling_strategy=SAMPLING_STRATEGY,
    use_augmentation=USE_AUGMENTATION,
    preprocessing_preset=PREPROCESSING_PRESET
)

print("DataLoaders created successfully.")
print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Sampling strategy: {SAMPLING_STRATEGY}")
print(f"Use augmentation: {USE_AUGMENTATION}")
print(f"Preprocessing preset: {PREPROCESSING_PRESET}")

print("\nTrain distribution:")
print(train_df[LABEL_COL].value_counts().sort_index())

print("\nValidation distribution:")
print(val_df[LABEL_COL].value_counts().sort_index())

print("\nTest distribution:")
print(test_df[LABEL_COL].value_counts().sort_index())

if class_weights is not None:
    print("\nClass weights:")
    print(class_weights)


In [ ]:
# ============================================================
# Batch sanity check
# Confirms that images and labels have the expected tensor shapes before training.
# ============================================================

if SHOW_DATA_PREVIEW:
    images, labels = next(iter(train_loader))

    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("First labels:", labels[:10])

# Expected:
# images: [BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE]
# labels: [BATCH_SIZE]

# ============================================================
# Visualize example images
# Quick visual QA: useful for catching wrong paths, wrong normalization assumptions, or broken labels.
# ============================================================

def denormalize_image(tensor):
    """
    Converts a normalized image tensor back to a displayable image.

    Input:
        tensor with shape [3, H, W]

    Output:
        image array with shape [H, W, 3]
    """

    imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    image = tensor.cpu() * imagenet_std + imagenet_mean
    image = image.clamp(0, 1)
    image = image.permute(1, 2, 0).numpy()

    return image


def show_batch(data_loader, num_images=8):
    """
    Shows a few images from a DataLoader.

    Input:
        data_loader: train_loader, val_loader or test_loader
        num_images: number of images to show
    """

    images, labels = next(iter(data_loader))

    num_images = min(num_images, len(images))

    plt.figure(figsize=(16, 6))

    for i in range(num_images):
        img = denormalize_image(images[i])

        plt.subplot(2, 4, i + 1)
        plt.imshow(img)
        plt.title(f"Label: {labels[i].item()}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


if SHOW_DATA_PREVIEW:
    show_batch(train_loader, num_images=8)
else:
    print("SHOW_DATA_PREVIEW is False. Skipping batch image preview for unattended matrix runs.")


## 3. Model And Training Utilities


In [ ]:
# ============================================================
# Model: ResNet baseline
# ResNet18 is a strong, simple CNN baseline for the fundus classification task.
# ============================================================

def create_classification_head(
    in_features,
    num_classes,
    dropout=RESNET_DROPOUT,
    use_custom_head=USE_CUSTOM_CLASSIFICATION_HEAD,
    hidden_dim=CUSTOM_HEAD_HIDDEN_DIM
):
    """
    Creates either the original simple ResNet classifier or a small custom MLP head.

    Simple head: optional Dropout -> Linear
    Custom head: Linear -> ReLU -> BatchNorm -> Dropout -> Linear
    """

    if use_custom_head:
        return nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    if dropout > 0:
        return nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_classes)
        )

    return nn.Linear(in_features, num_classes)


def set_backbone_trainable(model, trainable):
    """
    Freezes or unfreezes the pretrained ResNet backbone.

    The final classifier head is always left trainable, so USE_FROZEN_BACKBONE=True
    means "train only the head" rather than freezing the whole model.
    """

    for parameter in model.parameters():
        parameter.requires_grad = trainable

    for parameter in model.fc.parameters():
        parameter.requires_grad = True

    return model


def print_model_trainability(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")
    print(f"Frozen parameters:    {total_params - trainable_params:,}")


def create_model(
    num_classes,
    dropout=RESNET_DROPOUT,
    use_frozen_backbone=USE_FROZEN_BACKBONE,
    use_custom_head=USE_CUSTOM_CLASSIFICATION_HEAD,
    hidden_dim=CUSTOM_HEAD_HIDDEN_DIM
):
    """
    Creates a pretrained ResNet model and adapts it for classification.

    Defaults reproduce the original ResNet setup.
    Optional modes:
        - frozen backbone: train only the classifier head
        - custom head: replace the final linear layer with a small MLP head
    """

    model = models.resnet18(weights="IMAGENET1K_V1")

    in_features = model.fc.in_features
    model.fc = create_classification_head(
        in_features=in_features,
        num_classes=num_classes,
        dropout=dropout,
        use_custom_head=use_custom_head,
        hidden_dim=hidden_dim
    )

    if use_frozen_backbone:
        model = set_backbone_trainable(model, trainable=False)

    return model


model = create_model(
    NUM_CLASSES,
    dropout=RESNET_DROPOUT,
    use_frozen_backbone=USE_FROZEN_BACKBONE,
    use_custom_head=USE_CUSTOM_CLASSIFICATION_HEAD,
    hidden_dim=CUSTOM_HEAD_HIDDEN_DIM
)
model = model.to(DEVICE)
print_model_trainability(model)


# ============================================================
# Loss functions
# Supports standard, weighted, focal, and weighted focal losses for imbalance experiments.
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(
            logits,
            targets,
            weight=self.alpha,
            reduction="none"
        )
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()


def create_loss(loss_name=LOSS_NAME, class_weights=class_weights, device=DEVICE, focal_gamma=FOCAL_GAMMA):

    if class_weights is not None:
        class_weights = class_weights.to(device)

    if loss_name == "cross_entropy":
        return nn.CrossEntropyLoss()

    elif loss_name == "weighted_cross_entropy":
        if class_weights is None:
            raise ValueError("class_weights must be provided for weighted_cross_entropy")
        return nn.CrossEntropyLoss(weight=class_weights)

    elif loss_name == "focal_loss":
        return FocalLoss(gamma=focal_gamma)

    elif loss_name == "weighted_focal_loss":
        if class_weights is None:
            raise ValueError("class_weights must be provided for weighted_focal_loss")
        return FocalLoss(alpha=class_weights, gamma=focal_gamma)

    else:
        raise ValueError(f"Invalid LOSS_NAME: {loss_name}")


criterion = create_loss(
    loss_name=LOSS_NAME,
    class_weights=class_weights,
    device=DEVICE,
    focal_gamma=FOCAL_GAMMA
)


# ============================================================
# Optimizer and scheduler factories
# Keep optimizer choices configurable so Adam/AdamW/SGD comparisons are fair and repeatable.
# ============================================================

def create_optimizer(
    model,
    optimizer_name=OPTIMIZER_NAME,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    momentum=MOMENTUM
):
    """
    Creates optimizer based on config.
    """

    trainable_parameters = [
        parameter for parameter in model.parameters()
        if parameter.requires_grad
    ]

    if len(trainable_parameters) == 0:
        raise ValueError("No trainable parameters found. Check frozen-backbone settings.")

    if optimizer_name == "adam":
        return optim.Adam(
            trainable_parameters,
            lr=learning_rate,
            weight_decay=weight_decay
        )

    elif optimizer_name == "adamw":
        return optim.AdamW(
            trainable_parameters,
            lr=learning_rate,
            weight_decay=weight_decay
        )

    elif optimizer_name == "sgd":
        return optim.SGD(
            trainable_parameters,
            lr=learning_rate,
            momentum=momentum,
            weight_decay=weight_decay
        )

    else:
        raise ValueError(f"Invalid OPTIMIZER_NAME: {optimizer_name}")


def create_scheduler(optimizer, use_scheduler=USE_LR_SCHEDULER):
    """
    Reduces learning rate when validation Kappa stops improving.
    """

    if not use_scheduler:
        return None

    return optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=LR_SCHEDULER_FACTOR,
        patience=LR_SCHEDULER_PATIENCE,
        min_lr=MIN_LR
    )


optimizer = create_optimizer(model)
scheduler = create_scheduler(optimizer)


# ============================================================
# Metrics helper
# Kappa and macro F1 are more informative than accuracy for imbalanced ordinal DR severity labels.
# ============================================================

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred, weights="quadratic")
    }


def get_selection_score(metrics, metric_name=SELECTION_METRIC):
    """
    Returns the metric used for checkpointing, early stopping, scheduler updates, and Optuna.
    """

    if metric_name not in metrics:
        raise KeyError(f"Metric {metric_name} not found in metrics: {list(metrics.keys())}")

    return metrics[metric_name]


# ============================================================
# Train one epoch
# One focused training pass over the training loader.
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion):

    model.train()

    running_loss = 0.0

    for images, labels in tqdm(loader, desc="Training", leave=False):

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        logits = model(images)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)

    return epoch_loss

# ============================================================
# Evaluation helper
# Returns loss, metrics, predictions, and labels for reports, confusion matrices, and error analysis.
# ============================================================

def evaluate_detailed(model, loader, criterion):
    """
    Runs full evaluation and returns:
    - loss
    - metrics
    - predictions
    - labels

    Input:
        model
        loader
        criterion

    Output:
        loss, metrics, predictions, labels
    """

    model.eval()

    all_preds = []
    all_labels = []

    running_loss = 0.0

    with torch.no_grad():

        for images, labels in tqdm(loader, desc="Evaluating", leave=False):

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(images)

            loss = criterion(logits, labels)

            running_loss += loss.item() * images.size(0)

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    metrics = compute_metrics(all_labels, all_preds)

    return epoch_loss, metrics, all_preds, all_labels


In [ ]:
# ============================================================
# Training history and experiment tracking
# Stores per-epoch metrics locally and optionally mirrors them to Weights & Biases.
# ============================================================

history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_f1": [],
    "val_kappa": [],
    "learning_rate": [],
}


def update_history(history, train_loss, val_loss, metrics, learning_rate):
    """
    Stores metrics after each epoch.
    """

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(metrics["accuracy"])
    history["val_f1"].append(metrics["macro_f1"])
    history["val_kappa"].append(metrics["kappa"])
    history["learning_rate"].append(learning_rate)

    return history


def get_experiment_config():
    return {
        "experiment_name": EXPERIMENT_NAME,
        "model": "resnet18",
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "patience": PATIENCE,
        "selection_metric": SELECTION_METRIC,
        "loss_name": LOSS_NAME,
        "focal_gamma": FOCAL_GAMMA,
        "optimizer_name": OPTIMIZER_NAME,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "momentum": MOMENTUM,
        "resnet_dropout": RESNET_DROPOUT,
        "run_baseline_training": RUN_BASELINE_TRAINING,
        "run_final_evaluation": RUN_FINAL_EVALUATION,
        "run_gradcam": RUN_GRADCAM,
        "use_fundus_preprocessing": USE_FUNDUS_PREPROCESSING,
        "preprocessing_preset": PREPROCESSING_PRESET,
        "preprocess_crop_black_background": PREPROCESS_CROP_BLACK_BACKGROUND,
        "preprocess_autocontrast": PREPROCESS_AUTOCONTRAST,
        "use_frozen_backbone": USE_FROZEN_BACKBONE,
        "use_custom_classification_head": USE_CUSTOM_CLASSIFICATION_HEAD,
        "custom_head_hidden_dim": CUSTOM_HEAD_HIDDEN_DIM,
        "use_lr_scheduler": USE_LR_SCHEDULER,
        "use_augmentation": USE_AUGMENTATION,
        "sampling_strategy": SAMPLING_STRATEGY,
        "use_weighted_sampler": USE_WEIGHTED_SAMPLER,
        "use_weighted_loss": USE_WEIGHTED_LOSS,
        "val_size": VAL_SIZE,
        "seed": SEED,
    }


def init_wandb_run():
    if not USE_WANDB:
        return None

    try:
        import wandb
    except ImportError as exc:
        raise ImportError(
            "Weights & Biases is not installed. On Kaggle, run: !pip install wandb"
        ) from exc

    return wandb.init(
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        name=EXPERIMENT_NAME,
        config=get_experiment_config(),
        reinit=True
    )


def log_experiment_metrics(run, metrics):
    if run is not None:
        run.log(metrics)


def finish_wandb_run(run):
    if run is not None:
        run.finish()


# ============================================================
# Plot training curves
# Learning curves show overfitting/underfitting and whether the scheduler changed LR.
# ============================================================

def plot_training_curves(history):
    """
    Plots training and validation metrics over time.

    Input:
        history dictionary

    Output:
        matplotlib plots
    """

    epochs = range(1, len(history["train_loss"]) + 1)

    # ----------------------------
    # Loss curves
    # ----------------------------

    plt.figure(figsize=(8, 5))

    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Validation Loss")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()

    plt.show()

    # ----------------------------
    # Accuracy
    # ----------------------------

    plt.figure(figsize=(8, 5))

    plt.plot(epochs, history["val_accuracy"])

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Validation Accuracy")

    plt.show()

    # ----------------------------
    # Macro F1
    # ----------------------------

    plt.figure(figsize=(8, 5))

    plt.plot(epochs, history["val_f1"])

    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.title("Validation Macro F1")

    plt.show()

    # ----------------------------
    # Kappa
    # ----------------------------

    plt.figure(figsize=(8, 5))

    plt.plot(epochs, history["val_kappa"])

    plt.xlabel("Epoch")
    plt.ylabel("Quadratic Weighted Kappa")
    plt.title("Validation Kappa")

    plt.show()

    # ----------------------------
    # Learning rate
    # ----------------------------

    if "learning_rate" in history and len(history["learning_rate"]) > 0:
        plt.figure(figsize=(8, 5))

        plt.plot(epochs, history["learning_rate"])

        plt.xlabel("Epoch")
        plt.ylabel("Learning Rate")
        plt.title("Learning Rate")

        plt.show()


# ============================================================
# Confusion matrix visualization
# Important for seeing which DR grades are confused, not just overall score.
# ============================================================

def plot_confusion_matrix(y_true, y_pred, num_classes):
    """
    Plots confusion matrix.

    Input:
        y_true: real labels
        y_pred: predicted labels
        num_classes: number of classes

    Output:
        confusion matrix plot
    """

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=list(range(num_classes)),
        yticklabels=list(range(num_classes))
    )

    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.title("Confusion Matrix")

    plt.show()


## 4. Baseline Training


In [ ]:
# ============================================================
# Full training loop with history + early stopping
# The checkpoint criterion is configured via SELECTION_METRIC.
# ============================================================

if RUN_BASELINE_TRAINING:

    wandb_run = init_wandb_run()

    best_score = -1.0
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(NUM_EPOCHS):

        print("=" * 60)
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
        print("=" * 60)

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion
        )

        val_loss, val_metrics, val_preds, val_labels = evaluate_detailed(
            model,
            val_loader,
            criterion
        )

        current_score = get_selection_score(val_metrics)

        if scheduler is not None:
            scheduler.step(current_score)

        current_lr = optimizer.param_groups[0]["lr"]

        history = update_history(
            history,
            train_loss,
            val_loss,
            val_metrics,
            current_lr
        )

        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss:   {val_loss:.4f}")
        print()
        print(f"Val Accuracy: {val_metrics['accuracy']:.4f}")
        print(f"Val Macro F1: {val_metrics['macro_f1']:.4f}")
        print(f"Val Kappa:    {val_metrics['kappa']:.4f}")
        print(f"Selection Metric ({SELECTION_METRIC}): {current_score:.4f}")
        print(f"Learning Rate: {current_lr:.2e}")

        improved = current_score > best_score + MIN_DELTA

        if improved:

            best_score = current_score
            best_epoch = epoch + 1
            epochs_without_improvement = 0

            torch.save(
                model.state_dict(),
                BEST_MODEL_PATH
            )

            print("\nSaved new best model.")

        else:
            epochs_without_improvement += 1
            print(f"\nNo {SELECTION_METRIC} improvement for {epochs_without_improvement}/{PATIENCE} epochs.")

        log_experiment_metrics(
            wandb_run,
            {
                "epoch": epoch + 1,
                "train/loss": train_loss,
                "val/loss": val_loss,
                "val/accuracy": val_metrics["accuracy"],
                "val/macro_f1": val_metrics["macro_f1"],
                "val/kappa": val_metrics["kappa"],
                "optimization/lr": current_lr,
                f"early_stop/best_{SELECTION_METRIC}": best_score,
                "early_stop/no_improvement_epochs": epochs_without_improvement,
                "early_stop/improved": int(improved),
            }
        )

        print(f"Best Val {SELECTION_METRIC}: {best_score:.4f} at epoch {best_epoch}")
        print()

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break


    finish_wandb_run(wandb_run)

    print("Training finished.")
    print(f"Best Val {SELECTION_METRIC}: {best_score:.4f} at epoch {best_epoch}")

else:
    best_score = None
    best_epoch = None
    print("RUN_BASELINE_TRAINING is False. Skipping normal training.")


## 4b. R1-R7 Experiment Matrix

Runs all planned ResNet experiments, checkpointing by validation Macro-F1 and exporting CSV summaries for Kaggle download.


In [ ]:
# ============================================================
# R1-R7 experiment matrix runner
# Each run trains from a fresh ImageNet-pretrained ResNet18 and writes one summary row.
# ============================================================


def make_empty_history():
    return {
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_f1": [],
        "val_kappa": [],
        "learning_rate": [],
    }


def apply_run_augmentation_flags(run_config):
    """
    The transform factory reads these augmentation switches globally.
    Setting them before DataLoader creation keeps each run self-contained.
    """

    global USE_RANDOM_FLIP, USE_RANDOM_ROTATION, USE_COLOR_JITTER
    global USE_RANDOM_RESIZED_CROP, USE_RANDOM_ERASING

    USE_RANDOM_FLIP = run_config.get("use_random_flip", False)
    USE_RANDOM_ROTATION = run_config.get("use_random_rotation", False)
    USE_COLOR_JITTER = run_config.get("use_color_jitter", False)
    USE_RANDOM_RESIZED_CROP = run_config.get("use_random_resized_crop", False)
    USE_RANDOM_ERASING = run_config.get("use_random_erasing", False)


def flatten_classification_report(prefix, y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        output_dict=True,
        zero_division=0,
    )

    flat = {}

    for class_index in range(NUM_CLASSES):
        class_report = report[str(class_index)]
        flat[f"{prefix}_precision_class_{class_index}"] = class_report["precision"]
        flat[f"{prefix}_recall_class_{class_index}"] = class_report["recall"]
        flat[f"{prefix}_f1_class_{class_index}"] = class_report["f1-score"]
        flat[f"{prefix}_support_class_{class_index}"] = class_report["support"]

    flat[f"{prefix}_macro_precision"] = report["macro avg"]["precision"]
    flat[f"{prefix}_macro_recall"] = report["macro avg"]["recall"]
    flat[f"{prefix}_weighted_precision"] = report["weighted avg"]["precision"]
    flat[f"{prefix}_weighted_recall"] = report["weighted avg"]["recall"]
    flat[f"{prefix}_weighted_f1"] = report["weighted avg"]["f1-score"]

    return flat


def save_results_csv(summary_rows, epoch_rows):
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(RESULTS_CSV_PATH, index=False)

    epoch_df = pd.DataFrame(epoch_rows)
    epoch_df.to_csv(EPOCH_HISTORY_CSV_PATH, index=False)

    print(f"Saved summary CSV to: {RESULTS_CSV_PATH}")
    print(f"Saved epoch history CSV to: {EPOCH_HISTORY_CSV_PATH}")


def run_resnet_experiment(run_config):
    run_id = run_config["run_id"]
    run_name = run_config["experiment_name"]
    print("=" * 80)
    print(f"STARTING {run_id}: {run_name}")
    print("=" * 80)
    print(json.dumps(run_config, indent=2))

    set_seed(SEED)
    apply_run_augmentation_flags(run_config)

    batch_size = run_config.get("batch_size", BATCH_SIZE)
    selection_metric = run_config.get("selection_metric", SELECTION_METRIC)
    preprocessing_preset = run_config.get("preprocessing_preset", PREPROCESSING_PRESET)
    sampling_strategy = run_config.get("sampling_strategy", SAMPLING_STRATEGY)
    use_augmentation = run_config.get("use_augmentation", USE_AUGMENTATION)

    train_loader, val_loader, test_loader, run_class_weights, train_df_run, val_df_run = create_dataloaders(
        batch_size=batch_size,
        sampling_strategy=sampling_strategy,
        use_augmentation=use_augmentation,
        preprocessing_preset=preprocessing_preset,
    )

    run_model = create_model(
        NUM_CLASSES,
        dropout=run_config.get("resnet_dropout", RESNET_DROPOUT),
        use_frozen_backbone=run_config.get("use_frozen_backbone", USE_FROZEN_BACKBONE),
        use_custom_head=run_config.get("use_custom_classification_head", USE_CUSTOM_CLASSIFICATION_HEAD),
        hidden_dim=run_config.get("custom_head_hidden_dim", CUSTOM_HEAD_HIDDEN_DIM),
    ).to(DEVICE)

    print_model_trainability(run_model)

    run_criterion = create_loss(
        loss_name=run_config.get("loss_name", LOSS_NAME),
        class_weights=run_class_weights,
        device=DEVICE,
        focal_gamma=run_config.get("focal_gamma", FOCAL_GAMMA),
    )

    run_optimizer = create_optimizer(
        run_model,
        optimizer_name=run_config.get("optimizer_name", OPTIMIZER_NAME),
        learning_rate=run_config.get("learning_rate", LEARNING_RATE),
        weight_decay=run_config.get("weight_decay", WEIGHT_DECAY),
        momentum=run_config.get("momentum", MOMENTUM),
    )

    run_scheduler = create_scheduler(
        run_optimizer,
        use_scheduler=run_config.get("use_lr_scheduler", USE_LR_SCHEDULER),
    )

    best_model_path = f"/kaggle/working/{run_id.lower()}_best_resnet18.pth"
    history = make_empty_history()
    epoch_rows = []

    best_score = -1.0
    best_epoch = 0
    best_val_loss = None
    epochs_without_improvement = 0
    start_time = time.time()

    for epoch in range(run_config.get("num_epochs", NUM_EPOCHS)):
        print("-" * 70)
        print(f"{run_id} | Epoch {epoch + 1}/{run_config.get('num_epochs', NUM_EPOCHS)}")

        train_loss = train_one_epoch(run_model, train_loader, run_optimizer, run_criterion)
        val_loss, val_metrics, val_preds, val_labels = evaluate_detailed(run_model, val_loader, run_criterion)
        current_score = get_selection_score(val_metrics, metric_name=selection_metric)

        if run_scheduler is not None:
            run_scheduler.step(current_score)

        current_lr = run_optimizer.param_groups[0]["lr"]
        history = update_history(history, train_loss, val_loss, val_metrics, current_lr)

        epoch_rows.append({
            "run_id": run_id,
            "experiment_name": run_name,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_kappa": val_metrics["kappa"],
            "selection_metric": selection_metric,
            "selection_score": current_score,
            "learning_rate": current_lr,
        })

        print(
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
            f"Val Macro F1: {val_metrics['macro_f1']:.4f} | "
            f"Val Acc: {val_metrics['accuracy']:.4f} | "
            f"Val Kappa: {val_metrics['kappa']:.4f}"
        )

        improved = current_score > best_score + MIN_DELTA

        if improved:
            best_score = current_score
            best_epoch = epoch + 1
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save(run_model.state_dict(), best_model_path)
            print(f"Saved new best checkpoint for {run_id}: {best_score:.4f}")
        else:
            epochs_without_improvement += 1
            print(f"No {selection_metric} improvement for {epochs_without_improvement}/{run_config.get('patience', PATIENCE)} epochs.")

        if epochs_without_improvement >= run_config.get("patience", PATIENCE):
            print(f"Early stopping triggered for {run_id}.")
            break

    run_model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

    final_val_loss, final_val_metrics, final_val_preds, final_val_labels = evaluate_detailed(run_model, val_loader, run_criterion)
    test_loss, test_metrics, test_preds, test_labels = evaluate_detailed(run_model, test_loader, run_criterion)

    elapsed_minutes = (time.time() - start_time) / 60
    val_cm = confusion_matrix(final_val_labels, final_val_preds, labels=list(range(NUM_CLASSES))).tolist()
    test_cm = confusion_matrix(test_labels, test_preds, labels=list(range(NUM_CLASSES))).tolist()

    summary = {
        "run_id": run_id,
        "experiment_name": run_name,
        "model": "resnet18",
        "report_note": run_config.get("report_note", ""),
        "image_size": IMAGE_SIZE,
        "batch_size": batch_size,
        "num_workers": NUM_WORKERS,
        "num_epochs_configured": run_config.get("num_epochs", NUM_EPOCHS),
        "epochs_ran": len(history["train_loss"]),
        "patience": run_config.get("patience", PATIENCE),
        "min_delta": MIN_DELTA,
        "selection_metric": selection_metric,
        "seed": SEED,
        "val_size": VAL_SIZE,
        "use_stratified_split": USE_STRATIFIED_SPLIT,
        "device_mode": DEVICE_MODE,
        "device_used": str(DEVICE),
        "best_epoch": best_epoch,
        "best_selection_score": best_score,
        "best_val_loss_during_training": best_val_loss,
        "best_model_path": best_model_path,
        "use_fundus_preprocessing": USE_FUNDUS_PREPROCESSING,
        "preprocessing_preset": preprocessing_preset,
        "preprocess_crop_threshold": PREPROCESS_CROP_THRESHOLD,
        "preprocess_crop_padding": PREPROCESS_CROP_PADDING,
        "clahe_clip_limit": CLAHE_CLIP_LIMIT,
        "clahe_tile_grid_size": str(CLAHE_TILE_GRID_SIZE),
        "graham_blur_sigma": GRAHAM_BLUR_SIGMA,
        "graham_blend_alpha": GRAHAM_BLEND_ALPHA,
        "graham_blend_beta": GRAHAM_BLEND_BETA,
        "graham_blend_gamma": GRAHAM_BLEND_GAMMA,
        "sampling_strategy": sampling_strategy,
        "use_weighted_sampler": sampling_strategy == "weighted_oversample",
        "use_augmentation": use_augmentation,
        "use_random_flip": USE_RANDOM_FLIP,
        "use_random_rotation": USE_RANDOM_ROTATION,
        "use_color_jitter": USE_COLOR_JITTER,
        "use_random_resized_crop": USE_RANDOM_RESIZED_CROP,
        "use_random_erasing": USE_RANDOM_ERASING,
        "random_rotation_degrees": RANDOM_ROTATION_DEGREES,
        "random_resized_crop_scale": str(RANDOM_RESIZED_CROP_SCALE),
        "color_jitter_brightness": COLOR_JITTER_BRIGHTNESS,
        "color_jitter_contrast": COLOR_JITTER_CONTRAST,
        "color_jitter_saturation": COLOR_JITTER_SATURATION,
        "color_jitter_hue": COLOR_JITTER_HUE,
        "random_erasing_p": RANDOM_ERASING_P,
        "loss_name": run_config.get("loss_name", LOSS_NAME),
        "use_weighted_loss": run_config.get("loss_name", LOSS_NAME) in ["weighted_cross_entropy", "weighted_focal_loss"],
        "focal_gamma": run_config.get("focal_gamma", FOCAL_GAMMA),
        "optimizer_name": run_config.get("optimizer_name", OPTIMIZER_NAME),
        "learning_rate": run_config.get("learning_rate", LEARNING_RATE),
        "weight_decay": run_config.get("weight_decay", WEIGHT_DECAY),
        "momentum": run_config.get("momentum", MOMENTUM),
        "use_lr_scheduler": run_config.get("use_lr_scheduler", USE_LR_SCHEDULER),
        "lr_scheduler_factor": LR_SCHEDULER_FACTOR,
        "lr_scheduler_patience": LR_SCHEDULER_PATIENCE,
        "min_lr": MIN_LR,
        "use_frozen_backbone": run_config.get("use_frozen_backbone", USE_FROZEN_BACKBONE),
        "use_custom_classification_head": run_config.get("use_custom_classification_head", USE_CUSTOM_CLASSIFICATION_HEAD),
        "custom_head_hidden_dim": run_config.get("custom_head_hidden_dim", CUSTOM_HEAD_HIDDEN_DIM),
        "resnet_dropout": run_config.get("resnet_dropout", RESNET_DROPOUT),
        "pretrained_weights": "IMAGENET1K_V1",
        "train_samples": len(train_df_run),
        "val_samples": len(val_df_run),
        "test_samples": len(test_df),
        "elapsed_minutes": elapsed_minutes,
        "val_loss": final_val_loss,
        "val_accuracy": final_val_metrics["accuracy"],
        "val_macro_f1": final_val_metrics["macro_f1"],
        "val_kappa": final_val_metrics["kappa"],
        "test_loss": test_loss,
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_kappa": test_metrics["kappa"],
        "val_confusion_matrix_json": json.dumps(val_cm),
        "test_confusion_matrix_json": json.dumps(test_cm),
    }

    summary.update(flatten_classification_report("val", final_val_labels, final_val_preds))
    summary.update(flatten_classification_report("test", test_labels, test_preds))

    print("=" * 80)
    print(f"FINISHED {run_id}")
    print(f"Best epoch: {best_epoch}")
    print(f"Validation Macro F1: {final_val_metrics['macro_f1']:.4f}")
    print(f"Validation Kappa:    {final_val_metrics['kappa']:.4f}")
    print(f"Test Macro F1:       {test_metrics['macro_f1']:.4f}")
    print(f"Test Kappa:          {test_metrics['kappa']:.4f}")
    print("=" * 80)

    del run_model, run_optimizer, run_scheduler, train_loader, val_loader, test_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return summary, epoch_rows


if RUN_EXPERIMENT_MATRIX:
    all_summaries = []
    all_epoch_rows = []

    for run_config in EXPERIMENT_RUNS:
        summary, epoch_rows = run_resnet_experiment(run_config)
        all_summaries.append(summary)
        all_epoch_rows.extend(epoch_rows)
        save_results_csv(all_summaries, all_epoch_rows)

    results_df = pd.DataFrame(all_summaries).sort_values("val_macro_f1", ascending=False)
    display(results_df[[
        "run_id",
        "experiment_name",
        "preprocessing_preset",
        "sampling_strategy",
        "loss_name",
        "use_frozen_backbone",
        "use_augmentation",
        "best_epoch",
        "val_macro_f1",
        "val_kappa",
        "val_accuracy",
        "test_macro_f1",
        "test_kappa",
        "test_accuracy",
    ]])

    try:
        from IPython.display import FileLink
        display(FileLink(RESULTS_CSV_PATH))
        display(FileLink(EPOCH_HISTORY_CSV_PATH))
    except Exception:
        print(f"Download summary CSV from Kaggle output: {RESULTS_CSV_PATH}")
        print(f"Download epoch CSV from Kaggle output: {EPOCH_HISTORY_CSV_PATH}")
else:
    print("RUN_EXPERIMENT_MATRIX is False. Set it to True in the configuration cell to run R1-R7.")


## 5. Final Evaluation And Comparison Export


In [ ]:
if RUN_FINAL_EVALUATION:

    # ============================================================
    # Final evaluation and result export
    # The test set is evaluated only after selecting the best validation checkpoint.
    # ============================================================

    plot_training_curves(history)

    # ============================================================
    # 8. Load best model and evaluate validation set
    # ============================================================

    model.load_state_dict(
        torch.load(BEST_MODEL_PATH, map_location=DEVICE)
    )

    val_loss, val_metrics, val_preds, val_labels = evaluate_detailed(
        model,
        val_loader,
        criterion
    )

    print("=" * 60)
    print("BEST VALIDATION RESULTS")
    print("=" * 60)

    print(f"Validation Loss:     {val_loss:.4f}")
    print(f"Validation Accuracy: {val_metrics['accuracy']:.4f}")
    print(f"Validation Macro F1: {val_metrics['macro_f1']:.4f}")
    print(f"Validation Kappa:    {val_metrics['kappa']:.4f}")

    plot_confusion_matrix(
        y_true=val_labels,
        y_pred=val_preds,
        num_classes=NUM_CLASSES
    )

    print(
        classification_report(
            val_labels,
            val_preds,
            labels=list(range(NUM_CLASSES)),
            digits=4,
            zero_division=0
        )
    )

    # ============================================================
    # 9. Final test evaluation
    # ============================================================

    test_loss, test_metrics, test_preds, test_labels = evaluate_detailed(
        model,
        test_loader,
        criterion
    )

    print("=" * 60)
    print("FINAL TEST RESULTS")
    print("=" * 60)

    print(f"Test Loss:     {test_loss:.4f}")
    print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"Test Macro F1: {test_metrics['macro_f1']:.4f}")
    print(f"Test Kappa:    {test_metrics['kappa']:.4f}")

    plot_confusion_matrix(
        y_true=test_labels,
        y_pred=test_preds,
        num_classes=NUM_CLASSES
    )

    print(
        classification_report(
            test_labels,
            test_preds,
            labels=list(range(NUM_CLASSES)),
            digits=4,
            zero_division=0
        )
    )

    # ============================================================
    # Save one-row experiment summary for model/config comparison
    # This CSV becomes the comparison table across ResNet, ViT, losses, optimizers, and configs.
    # ============================================================

    summary_row = get_experiment_config()
    summary_row.update({
        f"best_val_{SELECTION_METRIC}": best_score,
        "best_epoch": best_epoch,
        "final_val_loss": val_loss,
        "final_val_accuracy": val_metrics["accuracy"],
        "final_val_macro_f1": val_metrics["macro_f1"],
        "final_val_kappa": val_metrics["kappa"],
        "test_loss": test_loss,
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_kappa": test_metrics["kappa"],
    })

    summary_df = pd.DataFrame([summary_row])

    if os.path.exists(RESULTS_CSV_PATH):
        old_summary_df = pd.read_csv(RESULTS_CSV_PATH)
        summary_df = pd.concat([old_summary_df, summary_df], ignore_index=True)

    summary_df.to_csv(RESULTS_CSV_PATH, index=False)
    print(f"Saved experiment summary to: {RESULTS_CSV_PATH}")
else:
    print("RUN_FINAL_EVALUATION is False. Skipping final validation/test evaluation.")


## 6. Interpretability


In [ ]:
# ============================================================
# Grad-CAM visualization for ResNet interpretability
# Helps check whether the CNN focuses on plausible retinal regions instead of shortcuts.
# ============================================================

def show_gradcam(model, dataset, index=0, target_class=None, alpha=0.45):
    """
    Visualizes which image regions influenced the ResNet prediction.

    Robust for frozen backbones: the input tensor is marked as requiring gradients,
    so gradients through the frozen feature extractor are still available for Grad-CAM.
    """

    model.eval()

    image, label = dataset[index]
    input_tensor = image.unsqueeze(0).to(DEVICE)
    input_tensor.requires_grad_(True)

    activations = []
    gradients = []

    def forward_hook(module, module_input, module_output):
        activations.append(module_output)
        module_output.register_hook(lambda grad: gradients.append(grad.detach()))

    forward_handle = model.layer4[-1].register_forward_hook(forward_hook)

    try:
        logits = model(input_tensor)
        pred_class = int(torch.argmax(logits, dim=1).item())

        if target_class is None:
            target_class = pred_class

        model.zero_grad(set_to_none=True)
        logits[0, target_class].backward()

    finally:
        forward_handle.remove()

    if len(activations) == 0 or len(gradients) == 0:
        print("Grad-CAM skipped: no activations/gradients were captured.")
        return

    feature_maps = activations[0].detach()
    grad_maps = gradients[0]
    weights = grad_maps.mean(dim=(2, 3), keepdim=True)

    cam = (weights * feature_maps).sum(dim=1).squeeze()
    cam = torch.relu(cam)
    cam = cam - cam.min()

    if cam.max() > 0:
        cam = cam / cam.max()

    cam = torch.nn.functional.interpolate(
        cam.unsqueeze(0).unsqueeze(0),
        size=(image.shape[1], image.shape[2]),
        mode="bilinear",
        align_corners=False
    ).squeeze().cpu().numpy()

    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    image_np = image.cpu().numpy()
    image_np = np.clip((image_np * std) + mean, 0, 1)
    image_np = np.transpose(image_np, (1, 2, 0))

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(image_np)
    plt.title(f"True: {label} | Pred: {pred_class}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(image_np)
    plt.imshow(cam, cmap="jet", alpha=alpha)
    plt.title(f"Grad-CAM class {target_class}")
    plt.axis("off")

    plt.show()


if RUN_GRADCAM:
    for example_index in range(min(3, len(test_loader.dataset))):
        show_gradcam(model, test_loader.dataset, index=example_index)
else:
    print("RUN_GRADCAM is False. Skipping Grad-CAM visualizations.")


## 8. Optional Single-Run Optuna

Disabled by default. For R1-R7, prefer the fixed matrix first; tune only the best family afterwards if time remains.


In [ ]:
# ============================================================
# Optional Optuna objective for ResNet hyperparameter tuning
# This variant maximizes validation Macro F1.
# ============================================================

try:
    import optuna
except ImportError:
    optuna = None


def objective(trial):
    """
    Tunes frozen-backbone/custom-head hyperparameters and maximizes validation Macro F1.

    The search is intentionally compact to conserve Kaggle GPU hours.
    """

    set_seed(SEED)

    learning_rate = trial.suggest_float("learning_rate", 8e-5, 2e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 3e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16])
    optimizer_name = trial.suggest_categorical("optimizer_name", ["adam", "adamw"])

    dropout = trial.suggest_float("dropout", 0.0, 0.70)
    use_scheduler = trial.suggest_categorical("use_lr_scheduler", [False, True])
    hidden_dim = trial.suggest_categorical("custom_head_hidden_dim", [256, 512, 768])

    sampling_strategy = trial.suggest_categorical(
        "sampling_strategy",
        ["none", "weighted_oversample", "undersample"]
    )

    use_augmentation = trial.suggest_categorical(
        "use_augmentation",
        [False, True]
    )

    loss_name = trial.suggest_categorical(
        "loss_name",
        [
            "cross_entropy",
            "weighted_cross_entropy",
            "focal_loss",
            "weighted_focal_loss"
        ]
    )

    focal_gamma = trial.suggest_float("focal_gamma", 1.0, 4.0)

    trial_train_loader, trial_val_loader, _, trial_class_weights, _, _ = create_dataloaders(
        batch_size=batch_size,
        sampling_strategy=sampling_strategy,
        use_augmentation=use_augmentation
    )

    trial_model = create_model(
        NUM_CLASSES,
        dropout=dropout,
        use_frozen_backbone=True,
        use_custom_head=True,
        hidden_dim=hidden_dim
    ).to(DEVICE)

    trial_criterion = create_loss(
        loss_name=loss_name,
        class_weights=trial_class_weights,
        device=DEVICE,
        focal_gamma=focal_gamma
    )

    trial_optimizer = create_optimizer(
        trial_model,
        optimizer_name=optimizer_name,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        momentum=MOMENTUM
    )

    trial_scheduler = create_scheduler(
        trial_optimizer,
        use_scheduler=use_scheduler
    )

    best_trial_score = -1.0
    best_trial_epoch = 0
    epochs_without_improvement = 0
    trial_model_path = f"/kaggle/working/best_resnet_macro_f1_trial_{trial.number}.pth"

    for epoch in range(OPTUNA_MAX_EPOCHS):

        train_loss = train_one_epoch(
            trial_model,
            trial_train_loader,
            trial_optimizer,
            trial_criterion
        )

        val_loss, val_metrics, _, _ = evaluate_detailed(
            trial_model,
            trial_val_loader,
            trial_criterion
        )

        val_score = get_selection_score(val_metrics, metric_name="macro_f1")

        if trial_scheduler is not None:
            trial_scheduler.step(val_score)

        trial.report(val_score, step=epoch)

        if optuna is not None and trial.should_prune():
            del trial_model
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()

        if val_score > best_trial_score + MIN_DELTA:
            best_trial_score = val_score
            best_trial_epoch = epoch + 1
            epochs_without_improvement = 0
            torch.save(trial_model.state_dict(), trial_model_path)
        else:
            epochs_without_improvement += 1

        print(
            f"Trial {trial.number} | Epoch {epoch + 1}/{OPTUNA_MAX_EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
            f"Val Macro F1: {val_metrics['macro_f1']:.4f} | "
            f"Val Acc: {val_metrics['accuracy']:.4f} | "
            f"Val Kappa: {val_metrics['kappa']:.4f} | Best F1: {best_trial_score:.4f}"
        )

        if epochs_without_improvement >= OPTUNA_PATIENCE:
            break

    trial.set_user_attr("best_epoch", best_trial_epoch)
    trial.set_user_attr("best_model_path", trial_model_path)

    del trial_model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return best_trial_score


In [ ]:
# ============================================================
# Run Optuna Study (optional)
# This cell does nothing unless RUN_OPTUNA=True in the config cell.
# ============================================================

if RUN_OPTUNA:
    if optuna is None:
        raise ImportError(
            "Optuna is not installed. On Kaggle, run this once before the Optuna section: !pip install optuna -q"
        )

    study = optuna.create_study(
        direction="maximize",
        study_name=f"resnet_macro_f1_tuning_{OPTUNA_N_TRIALS}_trials",
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=3,
            n_warmup_steps=3
        )
    )

    study.optimize(
        objective,
        n_trials=OPTUNA_N_TRIALS
    )

    print("=" * 60)
    print("BEST OPTUNA TRIAL")
    print("=" * 60)

    print(f"Trial number: {study.best_trial.number}")
    print(f"Best validation Macro F1: {study.best_value:.4f}")
    print(f"Best epoch: {study.best_trial.user_attrs.get('best_epoch')}")
    print(f"Best model path: {study.best_trial.user_attrs.get('best_model_path')}")

    print("\nBest hyperparameters:")
    for key, value in study.best_params.items():
        print(f"{key}: {value}")
else:
    print("RUN_OPTUNA is False. Set RUN_OPTUNA = True in the configuration cell to start tuning.")
